# Modello DRAEMFPN

## Configurazione

In [ ]:
import os
import torch
from PIL import Image
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import itertools
import math
import numpy as np
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import random
#Per memorizzare quanto tempo ci mette
!pip install ipython-autotime
%load_ext autotime


#PATH per i relativi dataset
MVTEC_ROOT = '/content/drive/MyDrive/Project Work CV/MVTec1'
DTD_ROOT = '/content/drive/MyDrive/Project Work CV/dtd/images'
SAVE_DIR = '/content/drive/MyDrive/Project Work CV/weights/fpn/'
LOAD_DIR = '/content/drive/MyDrive/Project Work CV/paper_weights/DRAEM_seg_large_ae_large_0.0001_800_bs8_'

#Categorie da codificare
LABELS = ['leather', 'pill', 'tile', 'transistor', 'wood']
#batch size di 8 per non saturare la GPU di Colab
global_batch_size = 8

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 65.3 MB/s eta 0:00:00
time: 557 µs (started: 2026-06-04 12:58:02 +00:00)


# Dataset

## Caricamento dataset MVTec da google drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

time: 57.2 s (started: 2026-06-04 12:58:02 +00:00)


In [ ]:
class RGB_to_BGR(object):
    def __call__(self, tensor):
        return tensor[[2,1,0], ...]

transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    RGB_to_BGR()
])

### Funzione getDataset

In [ ]:

#La funzione getDatasets restituisce la lista di dataset per label
def getDatasets(root_dir, labels, transform):

    dataset_list = []

    for label in labels:
        train_directory = os.path.join(root_dir, label, 'train')

        if os.path.exists(train_directory):
            dataset = datasets.ImageFolder(root=train_directory, transform=transform)
            dataset_list.append((label,dataset))

    return dataset_list


## Describable Textures Dataset (DTD)
Dataset utilizzato per prendere le texture di rumore

In [ ]:
texture_dataset = datasets.ImageFolder(root=DTD_ROOT, transform=transform)

# Rumore di Perlin

## Funzione generatrice rumore di perlin

In [ ]:
def lerp_np(x,y,w):
    fin_out = (y-x)*w + x
    return fin_out

def generate_fractal_noise_2d(shape, res, octaves=1, persistence=0.5):
    noise = np.zeros(shape)
    frequency = 1
    amplitude = 1
    for _ in range(octaves):
        noise += amplitude * generate_perlin_noise_2d(shape, (frequency*res[0], frequency*res[1]))
        frequency *= 2
        amplitude *= persistence
    return noise


def generate_perlin_noise_2d(shape, res):
    def f(t):
        return 6 * t ** 5 - 15 * t ** 4 + 10 * t ** 3

    delta = (res[0] / shape[0], res[1] / shape[1])
    d = (shape[0] // res[0], shape[1] // res[1])
    grid = np.mgrid[0:res[0]:delta[0], 0:res[1]:delta[1]].transpose(1, 2, 0) % 1
    # Gradients
    angles = 2 * np.pi * np.random.rand(res[0] + 1, res[1] + 1)
    gradients = np.dstack((np.cos(angles), np.sin(angles)))
    g00 = gradients[0:-1, 0:-1].repeat(d[0], 0).repeat(d[1], 1)
    g10 = gradients[1:, 0:-1].repeat(d[0], 0).repeat(d[1], 1)
    g01 = gradients[0:-1, 1:].repeat(d[0], 0).repeat(d[1], 1)
    g11 = gradients[1:, 1:].repeat(d[0], 0).repeat(d[1], 1)
    # Ramps
    n00 = np.sum(grid * g00, 2)
    n10 = np.sum(np.dstack((grid[:, :, 0] - 1, grid[:, :, 1])) * g10, 2)
    n01 = np.sum(np.dstack((grid[:, :, 0], grid[:, :, 1] - 1)) * g01, 2)
    n11 = np.sum(np.dstack((grid[:, :, 0] - 1, grid[:, :, 1] - 1)) * g11, 2)
    # Interpolation
    t = f(grid)
    n0 = n00 * (1 - t[:, :, 0]) + t[:, :, 0] * n10
    n1 = n01 * (1 - t[:, :, 0]) + t[:, :, 0] * n11
    return np.sqrt(2) * ((1 - t[:, :, 1]) * n0 + t[:, :, 1] * n1)


def rand_perlin_2d_np(shape, res, fade=lambda t: 6 * t ** 5 - 15 * t ** 4 + 10 * t ** 3):
    delta = (res[0] / shape[0], res[1] / shape[1])
    d = (shape[0] // res[0], shape[1] // res[1])
    grid = np.mgrid[0:res[0]:delta[0], 0:res[1]:delta[1]].transpose(1, 2, 0) % 1

    angles = 2 * math.pi * np.random.rand(res[0] + 1, res[1] + 1)
    gradients = np.stack((np.cos(angles), np.sin(angles)), axis=-1)
    tt = np.repeat(np.repeat(gradients,d[0],axis=0),d[1],axis=1)

    tile_grads = lambda slice1, slice2: np.repeat(np.repeat(gradients[slice1[0]:slice1[1], slice2[0]:slice2[1]],d[0],axis=0),d[1],axis=1)
    dot = lambda grad, shift: (
                np.stack((grid[:shape[0], :shape[1], 0] + shift[0], grid[:shape[0], :shape[1], 1] + shift[1]),
                            axis=-1) * grad[:shape[0], :shape[1]]).sum(axis=-1)

    n00 = dot(tile_grads([0, -1], [0, -1]), [0, 0])
    n10 = dot(tile_grads([1, None], [0, -1]), [-1, 0])
    n01 = dot(tile_grads([0, -1], [1, None]), [0, -1])
    n11 = dot(tile_grads([1, None], [1, None]), [-1, -1])
    t = fade(grid[:shape[0], :shape[1]])
    return math.sqrt(2) * lerp_np(lerp_np(n00, n10, t[..., 0]), lerp_np(n01, n11, t[..., 0]), t[..., 1])


def rand_perlin_2d(shape, res, fade=lambda t: 6 * t ** 5 - 15 * t ** 4 + 10 * t ** 3):
    delta = (res[0] / shape[0], res[1] / shape[1])
    d = (shape[0] // res[0], shape[1] // res[1])

    grid = torch.stack(torch.meshgrid(torch.arange(0, res[0], delta[0]), torch.arange(0, res[1], delta[1])), dim=-1) % 1
    angles = 2 * math.pi * torch.rand(res[0] + 1, res[1] + 1)
    gradients = torch.stack((torch.cos(angles), torch.sin(angles)), dim=-1)

    tile_grads = lambda slice1, slice2: gradients[slice1[0]:slice1[1], slice2[0]:slice2[1]].repeat_interleave(d[0],
                                                                                                              0).repeat_interleave(
        d[1], 1)
    dot = lambda grad, shift: (
                torch.stack((grid[:shape[0], :shape[1], 0] + shift[0], grid[:shape[0], :shape[1], 1] + shift[1]),
                            dim=-1) * grad[:shape[0], :shape[1]]).sum(dim=-1)

    n00 = dot(tile_grads([0, -1], [0, -1]), [0, 0])

    n10 = dot(tile_grads([1, None], [0, -1]), [-1, 0])
    n01 = dot(tile_grads([0, -1], [1, None]), [0, -1])
    n11 = dot(tile_grads([1, None], [1, None]), [-1, -1])
    t = fade(grid[:shape[0], :shape[1]])
    return math.sqrt(2) * torch.lerp(torch.lerp(n00, n10, t[..., 0]), torch.lerp(n01, n11, t[..., 0]), t[..., 1])


def rand_perlin_2d_octaves(shape, res, octaves=1, persistence=0.5):
    noise = torch.zeros(shape)
    frequency = 1
    amplitude = 1
    for _ in range(octaves):
        noise += amplitude * rand_perlin_2d(shape, (frequency * res[0], frequency * res[1]))
        frequency *= 2
        amplitude *= persistence
    return noise

In [ ]:
#funzione di perlin costum per applicare il rumore all'immagine durante il training
def applyPerlinNoise(images, texture_images,label_name):
    batch_size, channel, height, widht = images.shape
    device = images.device

    augmented_images = torch.zeros_like(images)
    anomaly_masks = torch.zeros((batch_size, 1, height, widht), device=device)

    for i in range(batch_size):
        #applico il rumore all'immagine con probabilità del 50%
        if torch.rand(1).item() > 0.5:
            augmented_images[i] = images[i]
            continue

        #generazione immagine di perlin casuale
        perlin_x = 2 ** torch.randint(0, 6, (1,)).item()
        perlin_y = 2 ** torch.randint(0, 6, (1,)).item()

        height, widht = images.shape[2:]

        perlin_image = rand_perlin_2d((height,widht),(perlin_x, perlin_y))
        perlin_image = perlin_image.to(device)

        #soglia di decisione per la maschera Ma
        treshold = torch.rand(1).item()

        mask = (perlin_image > treshold).float().unsqueeze(0)

        '''
        applicazione del rumore di perlin se l'immagine si trova nella categoria oggetto
        if(label_name in L_OBJECT):
            fg_mask = (images[i].sum(dim=0) > 0.8).float().unsqueeze(0)
            mask = mask * fg_mask
        '''
        #beta valore compreso tra 0.1 e 1.0 come nel paper
        beta = (torch.rand(1, device=device)*0.9) + 0.1


        img = images[i]
        texture = texture_images[i]

        #formula immagine Ia
        augmented_image = img * (1 - mask) + \
                          beta * (texture * mask) + \
                          (1 - beta) * (img * mask)

        augmented_images[i] = augmented_image
        anomaly_masks[i] = mask

    return augmented_images, anomaly_masks




# Modello DRAEM - Feature Pyramid Network

## AutoEncoder

In [ ]:
class EncoderReconstructive(nn.Module):
    def __init__(self, in_channels, base_width):
        super(EncoderReconstructive, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels,base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True))
        self.mp1 = nn.Sequential(nn.MaxPool2d(2))
        self.block2 = nn.Sequential(
            nn.Conv2d(base_width,base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*2, base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True))
        self.mp2 = nn.Sequential(nn.MaxPool2d(2))
        self.block3 = nn.Sequential(
            nn.Conv2d(base_width*2,base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*4, base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True))
        self.mp3 = nn.Sequential(nn.MaxPool2d(2))
        self.block4 = nn.Sequential(
            nn.Conv2d(base_width*4,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))
        self.mp4 = nn.Sequential(nn.MaxPool2d(2))
        self.block5 = nn.Sequential(
            nn.Conv2d(base_width*8,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))


    def forward(self, x):
        b1 = self.block1(x)
        mp1 = self.mp1(b1)
        b2 = self.block2(mp1)
        mp2 = self.mp3(b2)
        b3 = self.block3(mp2)
        mp3 = self.mp3(b3)
        b4 = self.block4(mp3)
        mp4 = self.mp4(b4)
        b5 = self.block5(mp4)
        return b5


class DecoderReconstructive(nn.Module):
    def __init__(self, base_width, out_channels=1):
        super(DecoderReconstructive, self).__init__()

        self.up1 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width * 8, base_width * 8, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width * 8),
                                 nn.ReLU(inplace=True))
        self.db1 = nn.Sequential(
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width * 8, base_width * 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width * 4),
            nn.ReLU(inplace=True)
        )

        self.up2 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width * 4, base_width * 4, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width * 4),
                                 nn.ReLU(inplace=True))
        self.db2 = nn.Sequential(
            nn.Conv2d(base_width*4, base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width * 4, base_width * 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width * 2),
            nn.ReLU(inplace=True)
        )

        self.up3 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width * 2, base_width*2, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width*2),
                                 nn.ReLU(inplace=True))
        # cat with base*1
        self.db3 = nn.Sequential(
            nn.Conv2d(base_width*2, base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*2, base_width*1, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*1),
            nn.ReLU(inplace=True)
        )

        self.up4 = nn.Sequential(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True),
                                 nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
                                 nn.BatchNorm2d(base_width),
                                 nn.ReLU(inplace=True))
        self.db4 = nn.Sequential(
            nn.Conv2d(base_width*1, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True)
        )

        self.fin_out = nn.Sequential(nn.Conv2d(base_width, out_channels, kernel_size=3, padding=1))
        #self.fin_out = nn.Conv2d(base_width, out_channels, kernel_size=3, padding=1)

    def forward(self, b5):
        up1 = self.up1(b5)
        db1 = self.db1(up1)

        up2 = self.up2(db1)
        db2 = self.db2(up2)

        up3 = self.up3(db2)
        db3 = self.db3(up3)

        up4 = self.up4(db3)
        db4 = self.db4(up4)

        out = self.fin_out(db4)
        return out

#Wrapper AutoEncoder
#Classe AutoEncoder
class ReconstructiveSubNetwork(nn.Module):
    def __init__(self,in_channels=3, out_channels=3, base_width=128):
        super(ReconstructiveSubNetwork, self).__init__()
        self.encoder = EncoderReconstructive(in_channels, base_width)
        self.decoder = DecoderReconstructive(base_width, out_channels=out_channels)

    def forward(self, x):
        b5 = self.encoder(x)
        output = self.decoder(b5)
        return output


## Feature Pyramid Network

In [ ]:
class EncoderDiscriminative(nn.Module):
    def __init__(self, in_channels, base_width):
        super(EncoderDiscriminative, self).__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels,base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width, base_width, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width),
            nn.ReLU(inplace=True))
        self.mp1 = nn.Sequential(nn.MaxPool2d(2))
        self.block2 = nn.Sequential(
            nn.Conv2d(base_width,base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*2, base_width*2, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*2),
            nn.ReLU(inplace=True))
        self.mp2 = nn.Sequential(nn.MaxPool2d(2))
        self.block3 = nn.Sequential(
            nn.Conv2d(base_width*2,base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*4, base_width*4, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*4),
            nn.ReLU(inplace=True))
        self.mp3 = nn.Sequential(nn.MaxPool2d(2))
        self.block4 = nn.Sequential(
            nn.Conv2d(base_width*4,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))
        self.mp4 = nn.Sequential(nn.MaxPool2d(2))
        self.block5 = nn.Sequential(
            nn.Conv2d(base_width*8,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))

        self.mp5 = nn.Sequential(nn.MaxPool2d(2))
        self.block6 = nn.Sequential(
            nn.Conv2d(base_width*8,base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True),
            nn.Conv2d(base_width*8, base_width*8, kernel_size=3, padding=1),
            nn.BatchNorm2d(base_width*8),
            nn.ReLU(inplace=True))


    def forward(self, x):
        b1 = self.block1(x)
        mp1 = self.mp1(b1)
        b2 = self.block2(mp1)
        mp2 = self.mp2(b2)
        b3 = self.block3(mp2)
        mp3 = self.mp3(b3)
        b4 = self.block4(mp3)
        mp4 = self.mp4(b4)
        b5 = self.block5(mp4)
        mp5 = self.mp5(b5)
        b6 = self.block6(mp5)
        return b1,b2,b3,b4,b5,b6


#6 tensori in uscita come i blocchi dell'encoder
#non devo generare le mappe di feature a diversa scala perchè uso già quelle dell'encoder
#quindi parto dalle convoluzione laterali per ridimensionare in profondità
class DecoderFPN(nn.Module):
    def __init__(self, base_width=64, fpn_channels=256, out_channels=2):
        super(DecoderFPN, self).__init__()
        #conv 1x1
        self.depth_conv6 = nn.Conv2d(base_width * 8, fpn_channels, kernel_size=1)
        self.depth_conv5 = nn.Conv2d(base_width * 8, fpn_channels, kernel_size=1)
        self.depth_conv4 = nn.Conv2d(base_width * 8, fpn_channels, kernel_size=1)
        self.depth_conv3 = nn.Conv2d(base_width * 4, fpn_channels, kernel_size=1)
        self.depth_conv2 = nn.Conv2d(base_width * 2, fpn_channels, kernel_size=1)
        self.depth_conv1 = nn.Conv2d(base_width * 1, fpn_channels, kernel_size=1)

        #convluzioni 3x3 di smoothing per eliminare errori di upsampling
        self.smooth6 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)
        self.smooth5 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)
        self.smooth4 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)
        self.smooth3 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)
        self.smooth2 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)
        self.smooth1 = nn.Conv2d(fpn_channels, fpn_channels, kernel_size=3, padding=1)

        #Output
        #fpn_channels * 6 perchè riceve sei mappe di feature
        self.fin_conv = nn.Sequential(
            nn.Conv2d(fpn_channels * 6, fpn_channels, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(fpn_channels, out_channels, kernel_size=3, padding=1)
        )

    def forward(self, b1, b2, b3, b4, b5, b6):

        #Somma mappe di feature + smoothin 3x3
        p6 = self.depth_conv6(b6)

        p5 = self.depth_conv5(b5) + F.interpolate(p6, scale_factor=2, mode='bilinear', align_corners=False)
        p5 = self.smooth5(p5)

        p4 = self.depth_conv4(b4) + F.interpolate(p5, scale_factor=2, mode='bilinear', align_corners=False)
        p4 = self.smooth4(p4)

        p3 = self.depth_conv3(b3) + F.interpolate(p4, scale_factor=2, mode='bilinear', align_corners=False)
        p3 = self.smooth3(p3)

        p2 = self.depth_conv2(b2) + F.interpolate(p3, scale_factor=2, mode='bilinear', align_corners=False)
        p2 = self.smooth2(p2)

        p1 = self.depth_conv1(b1) + F.interpolate(p2, scale_factor=2, mode='bilinear', align_corners=False)
        p1 = self.smooth1(p1)

        #Upsample a 256x256
        original_size = (256,256)

        p6_up = F.interpolate(p6, size=original_size, mode='bilinear', align_corners=False)
        p5_up = F.interpolate(p5, size=original_size, mode='bilinear', align_corners=False)
        p4_up = F.interpolate(p4, size=original_size, mode='bilinear', align_corners=False)
        p3_up = F.interpolate(p3, size=original_size, mode='bilinear', align_corners=False)
        p2_up = F.interpolate(p2, size=original_size, mode='bilinear', align_corners=False)
        p1_up = F.interpolate(p1, size=original_size, mode='bilinear', align_corners=False)

        #Concatenazione mappe
        out_concat = torch.cat((p1_up,p2_up,p3_up,p4_up,p5_up,p6_up),dim=1)

        out = self.fin_conv(out_concat)

        return out


class FPNNetwork(nn.Module):
    def __init__(self, in_channels=6, out_channels=2, base_width=64):
        super(FPNNetwork, self).__init__()
        self.encoder = EncoderDiscriminative(in_channels, base_width)
        self.decoder = DecoderFPN(base_width, fpn_channels=256, out_channels=out_channels)

    def forward(self, x):
        b1, b2, b3, b4, b5, b6 = self.encoder(x)
        fpn_output = self.decoder(b1,b2,b3,b4,b5,b6)
        return fpn_output



## Inizializzazione pesi per FPN
I pesi dell'AutoEncoder li carico dal paper

In [ ]:
#formula rivisitata del paper
def weights_init(layer):
    #Si trova nel layer convoluzionale
    if isinstance(layer, nn.Conv2d):
        nn.init.normal_(layer.weight.data, mean=0.0, std=0.02)
    elif isinstance(layer, nn.BatchNorm2d):
        nn.init.normal_(layer.weight.data, mean=1.0, std=0.02)
        nn.init.constant_(layer.bias.data, 0.0)

## Loss di training:
- Focal Loss
- SSIM Loss

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from math import exp

class FocalLoss(nn.Module):
    """
    copy from: https://github.com/Hsuxu/Loss_ToolBox-PyTorch/blob/master/FocalLoss/FocalLoss.py
    This is a implementation of Focal Loss with smooth label cross entropy supported which is proposed in
    'Focal Loss for Dense Object Detection. (https://arxiv.org/abs/1708.02002)'
        Focal_Loss= -1*alpha*(1-pt)*log(pt)
    :param alpha: (tensor) 3D or 4D the scalar factor for this criterion
    :param gamma: (float,double) gamma > 0 reduces the relative loss for well-classified examples (p>0.5) putting more
                    focus on hard misclassified example
    :param smooth: (float,double) smooth value when cross entropy
    :param balance_index: (int) balance class index, should be specific when alpha is float
    :param size_average: (bool, optional) By default, the losses are averaged over each loss element in the batch.
    """

    def __init__(self, apply_nonlin=None, alpha=None, gamma=2, balance_index=0, smooth=1e-5, size_average=True):
        super(FocalLoss, self).__init__()
        self.apply_nonlin = apply_nonlin
        self.alpha = alpha
        self.gamma = gamma
        self.balance_index = balance_index
        self.smooth = smooth
        self.size_average = size_average

        if self.smooth is not None:
            if self.smooth < 0 or self.smooth > 1.0:
                raise ValueError('smooth value should be in [0,1]')

    def forward(self, logit, target):
        if self.apply_nonlin is not None:
            logit = self.apply_nonlin(logit)
        num_class = logit.shape[1]

        if logit.dim() > 2:
            # N,C,d1,d2 -> N,C,m (m=d1*d2*...)
            logit = logit.view(logit.size(0), logit.size(1), -1)
            logit = logit.permute(0, 2, 1).contiguous()
            logit = logit.view(-1, logit.size(-1))
        target = torch.squeeze(target, 1)
        target = target.view(-1, 1)
        alpha = self.alpha

        if alpha is None:
            alpha = torch.ones(num_class, 1)
        elif isinstance(alpha, (list, np.ndarray)):
            assert len(alpha) == num_class
            alpha = torch.FloatTensor(alpha).view(num_class, 1)
            alpha = alpha / alpha.sum()
        elif isinstance(alpha, float):
            alpha = torch.ones(num_class, 1)
            alpha = alpha * (1 - self.alpha)
            alpha[self.balance_index] = self.alpha

        else:
            raise TypeError('Not support alpha type')

        if alpha.device != logit.device:
            alpha = alpha.to(logit.device)

        idx = target.cpu().long()

        one_hot_key = torch.FloatTensor(target.size(0), num_class).zero_()
        one_hot_key = one_hot_key.scatter_(1, idx, 1)
        if one_hot_key.device != logit.device:
            one_hot_key = one_hot_key.to(logit.device)

        if self.smooth:
            one_hot_key = torch.clamp(
                one_hot_key, self.smooth / (num_class - 1), 1.0 - self.smooth)
        pt = (one_hot_key * logit).sum(1) + self.smooth
        logpt = pt.log()

        gamma = self.gamma

        alpha = alpha[idx]
        alpha = torch.squeeze(alpha)
        loss = -1 * alpha * torch.pow((1 - pt), gamma) * logpt

        if self.size_average:
            loss = loss.mean()
        return loss

def gaussian(window_size, sigma):
    gauss = torch.Tensor([exp(-(x - window_size//2)**2/float(2*sigma**2)) for x in range(window_size)])
    return gauss/gauss.sum()

def create_window(window_size, channel=1):
    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    window = _2D_window.expand(channel, 1, window_size, window_size).contiguous()
    return window

def ssim(img1, img2, window_size=11, window=None, size_average=True, full=False, val_range=None):
    if val_range is None:
        if torch.max(img1) > 128:
            max_val = 255
        else:
            max_val = 1

        if torch.min(img1) < -0.5:
            min_val = -1
        else:
            min_val = 0
        l = max_val - min_val
    else:
        l = val_range

    padd = window_size//2
    (_, channel, height, width) = img1.size()
    if window is None:
        real_size = min(window_size, height, width)
        window = create_window(real_size, channel=channel).to(img1.device)

    mu1 = F.conv2d(img1, window, padding=padd, groups=channel)
    mu2 = F.conv2d(img2, window, padding=padd, groups=channel)

    mu1_sq = mu1.pow(2)
    mu2_sq = mu2.pow(2)
    mu1_mu2 = mu1 * mu2

    sigma1_sq = F.conv2d(img1 * img1, window, padding=padd, groups=channel) - mu1_sq
    sigma2_sq = F.conv2d(img2 * img2, window, padding=padd, groups=channel) - mu2_sq
    sigma12 = F.conv2d(img1 * img2, window, padding=padd, groups=channel) - mu1_mu2

    c1 = (0.01 * l) ** 2
    c2 = (0.03 * l) ** 2

    v1 = 2.0 * sigma12 + c2
    v2 = sigma1_sq + sigma2_sq + c2
    cs = torch.mean(v1 / v2)  # contrast sensitivity

    ssim_map = ((2 * mu1_mu2 + c1) * v1) / ((mu1_sq + mu2_sq + c1) * v2)

    if size_average:
        ret = ssim_map.mean()
    else:
        ret = ssim_map.mean(1).mean(1).mean(1)

    if full:
        return ret, cs
    return ret, ssim_map


class SSIM(torch.nn.Module):
    def __init__(self, window_size=11, size_average=True, val_range=None):
        super(SSIM, self).__init__()
        self.window_size = window_size
        self.size_average = size_average
        self.val_range = val_range

        # Assume 1 channel for SSIM
        self.channel = 1
        self.window = create_window(window_size)

    def forward(self, img1, img2):
        (_, channel, _, _) = img1.size()

        if channel == self.channel and self.window.dtype == img1.dtype and self.window.device == img1.device:
            window = self.window
        else:
            window = create_window(self.window_size, channel).to(img1.device).type(img1.dtype)
            self.window = window
            self.channel = channel

        s_score, ssim_map = ssim(img1, img2, window=window, window_size=self.window_size, size_average=self.size_average)
        return 1.0 - s_score

# Training del Modello

## Setup di training

In [ ]:
#Ottengo i dataset per ogni categoria
dataset_list = getDatasets(MVTEC_ROOT,LABELS,transform)

#check di cuda
if torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'

print("using device: ",device)

#parametri di training
lr = 0.0001
num_epoch = 10

l2_loss = nn.MSELoss()
ssim_loss = SSIM()
focal_loss = FocalLoss()


## Load dei pesi AutoEncoder

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo


italy_tz = ZoneInfo("Europe/Rome")
italy_time = datetime.now(italy_tz)
italy_format = italy_time.strftime('%d_%m-%H:%M')
print(italy_format)

def loadWeight(label_name, model_name):
    if (model_name == 'ae'):
        model = ReconstructiveSubNetwork(in_channels=3, out_channels=3, base_width=128).to(device)
        ae_filename = LOAD_DIR + label_name + '_.pckl'
        if os.path.exists(ae_filename):
            model.load_state_dict(torch.load(ae_filename, map_location=device))
            print(f"✅ Pesi per la categoria '{ae_filename}' caricati correttamente!")
        else:
            print(f"❌ Errore: File dei pesi non trovati. Verifica i percorsi:\n - {ae_filename}")

        return model

def saveWeights(model_seg, label_name):

    fpn_filename = f'{num_epoch}_{italy_format}_fpn_weights_{label_name}.pth'
    fpn_save_path = os.path.join(SAVE_DIR, fpn_filename)
    torch.save(model_seg.state_dict(), fpn_save_path)
    print(f"Pesi FPN salvati con successo in: {fpn_save_path}")

## Training loop con AMP

In [ ]:
import gc

dataset_list = getDatasets(MVTEC_ROOT,LABELS,transform)

for label_name, train_dataset in dataset_list:
    if(device=='cuda'):
        train_loader = DataLoader(train_dataset,batch_size=global_batch_size,shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
        texture_loader = DataLoader(texture_dataset,batch_size=global_batch_size, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    else:
        train_loader = DataLoader(train_dataset,batch_size=global_batch_size, shuffle=True, drop_last=True)
        texture_loader = DataLoader(texture_dataset,batch_size=global_batch_size, shuffle=True, drop_last=True)

    print(f"Addestramento sul dataset: {label_name}\n")


    model = loadWeight(label_name, 'ae')
    model.to(device)
    model_seg = FPNNetwork(in_channels=6, out_channels=2, base_width=64)
    model_seg.to(device)

    model_seg.apply(weights_init)


    optimizer = torch.optim.Adam(model_seg.parameters(), lr)

    model.eval()
    for param in model.parameters():
        param.requires_grad = False

    model_seg.train()

    texture_iterator = iter(itertools.cycle(texture_loader))

    for epoch in range(num_epoch):
        total_loss = 0.0

        scaler = torch.amp.GradScaler()

        for images, _ in train_loader:
            original_img = images.to(device)
            texture_batch, _ = next(texture_iterator)
            texture_batch = texture_batch.to(device)

            noisy_img, gt_mask = applyPerlinNoise(original_img, texture_batch, label_name)
            noisy_img = noisy_img.to(device)
            gt_mask = gt_mask.to(device)

            optimizer.zero_grad()

            with torch.amp.autocast(device_type='cuda'):

              rec_img = model(noisy_img)

              concat_img = torch.cat((rec_img, noisy_img),dim=1)

              out_mask = model_seg(concat_img)
              out_mask = torch.softmax(out_mask, dim=1)

              loss_focal = focal_loss(out_mask, gt_mask)

              loss = loss_focal


            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()

        print(f"Epoca {epoch+1}, loss {loss:.4f}")

    saveWeights(model_seg, label_name)

    del(model)
    del(model_seg)
    del(scaler)

    gc.collect()

    torch.cuda.empty_cache()




## Training loop senza AMP

In [ ]:
import gc

dataset_list = getDatasets(MVTEC_ROOT,LABELS,transform)

for label_name, train_dataset in dataset_list:

    if(device=='cuda'):
        train_loader = DataLoader(train_dataset,batch_size=global_batch_size,shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
        texture_loader = DataLoader(texture_dataset,batch_size=global_batch_size, shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    else:
        train_loader = DataLoader(train_dataset,batch_size=global_batch_size, shuffle=True, drop_last=True)
        texture_loader = DataLoader(texture_dataset,batch_size=global_batch_size, shuffle=True, drop_last=True)

    print(f"Addestramento sul dataset: {label_name}\n")

    #Un modello per ogni categoria (come nel paper)
    model = loadWeight(label_name, 'ae')
    model.to(device)
    model_seg = FPNNetwork(in_channels=6, out_channels=2, base_width=64)
    model_seg.to(device)

    model_seg.apply(weights_init)

    optimizer = torch.optim.Adam(model_seg.parameters(), lr)

    model.eval()
    for param in model.parameters():
        param.requires_grad = False

    model_seg.train()

    texture_iterator = iter(itertools.cycle(texture_loader))

    for epoch in range(num_epoch):
        total_loss = 0.0

        for images, _ in train_loader:
            original_img = images.to(device)
            texture_batch, _ = next(texture_iterator)
            texture_batch = texture_batch.to(device)

            noisy_img, gt_mask = applyPerlinNoise(original_img, texture_batch, label_name)
            noisy_img = noisy_img.to(device)
            gt_mask = gt_mask.to(device)

            optimizer.zero_grad()

            rec_img = model(noisy_img)

            concat_img = torch.cat((rec_img, noisy_img),dim=1)

            out_mask = model_seg(concat_img)
            out_mask = torch.softmax(out_mask, dim=1)

            loss_focal = focal_loss(out_mask, gt_mask)

            loss = loss_focal

            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoca {epoch+1}, loss {loss:.4f}")

    #saveWeights(model_seg, label_name)

    del(model)
    del(model_seg)
    del(scaler)

    gc.collect()

    torch.cuda.empty_cache()


